In [7]:
import streamlit as st
from langchain_community.document_loaders import WebBaseLoader
import bs4
from langchain.text_splitter import RecursiveCharacterTextSplitter
import os
from langchain_google_genai import GoogleGenerativeAIEmbeddings
import google.generativeai as genai
from langchain.vectorstores import FAISS
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.chains.question_answering import load_qa_chain
from langchain.prompts import PromptTemplate
from dotenv import load_dotenv

In [8]:
load_dotenv()
genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))

In [9]:
def get_text():
    urls = ["https://www.hedigital.tech/", 
            "https://www.hedigital.tech/about/", 
            "https://www.hedigital.tech/contact/", 
            "https://www.hedigital.tech/career/",
            "https://www.hedigital.tech/solution/ai-based/", 
            "https://www.hedigital.tech/solution/ooh-monitoring/", 
            "https://www.hedigital.tech/solution/digital-kyc/", 
            "https://www.hedigital.tech/solution/nlp/", 
            "https://www.hedigital.tech/solution/sales/", 
            "https://www.hedigital.tech/solution/mdm/", 
            ]

    all_text_documents = []
    for url in urls:
        loader = WebBaseLoader(web_paths=(url,), bs_kwargs=dict(parse_only=bs4.SoupStrainer(
                class_=("hero-container flex-col items-center justify-center text-white",
                        "container py-16 text-center",
                        "text-center text-primary text-2xl md:text-3xl lg:text-4xl font-bold",
                        "block lg:hidden",
                        "bg-[#101010] mission-container mt-10",
                        "container my-20",
                        "md:w-2/5 md:justify-center",
                        "lg:grid lg:grid-cols-2 gap-4",
                        "md:flex flex-row justify-between mt-10 gap-5 md:gap-16",
                        "md:w-1/2 w-full mt-8",
                        "text-[#F4D400] text-xl  uppercase font-bold ",
                        "m-0 py-2 text-justify font-normal text-[#ABABAB] text-xs md:text-sm",
                        "text-[#F4D400] text-xl  uppercase font-bold ",
                        "m-0 py-2 text-justify font-normal text-[#ABABAB] text-xs md:text-sm",
                        "w-full text-center my-10 md:my-16",
                        "pt-10 px-3 space-y-1",
                        "md:w-1/2 w-full space-y-8 mt-10",
                        "container md:px-20 grid md:grid-cols-2 justify-between items-center gap-6",
                        "bg-[#181818] p-6 rounded-xl shadow-md",
                        "container mx-auto my-2 xl:px-14 py-10",
                        "container",
                        "mx-auto my-10",
                        "container mx-auto py-10 md:py-20 md:flex md:flex-row justify-center items-center gap-10",
                        "jsx-1e4fbc5ed7de5c2e grid grid-cols-1 md:grid-cols-2 justify-center items-center gap-10",
                        )
            ))
        )
        text_documents = loader.load()
        all_text_documents.extend(text_documents)
    
    return all_text_documents

In [10]:
def get_text_chunks():
    text_documents = get_text()
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=10000, chunk_overlap=1000)
    documents = text_splitter.split_documents(text_documents)
    texts = [doc.page_content for doc in documents]
    return texts

In [11]:
def get_vector_store():
    text_chunks = get_text_chunks()
    embeddings = GoogleGenerativeAIEmbeddings(model="models/embedding-001")
    vector_store = FAISS.from_texts(text_chunks, embedding=embeddings)
    vector_store.save_local("faiss_index")

In [12]:
get_vector_store()